In [1]:
import cv2
import pickle

recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read("face_recognizer.yml")

with open("labels.pickle", "rb") as f:
    label_map = pickle.load(f)
id_to_name = {v: k for k, v in label_map.items()}

face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")
print("Model and labels loaded:", id_to_name)

Model and labels loaded: {0: 'Atul_Singh', 1: 'Bhanu_Yadav', 2: 'Piyush'}


In [2]:
import csv
import os
from datetime import datetime

CONFIDENCE_THRESHOLD = 55

def mark_attendance(name):
    today = datetime.now().strftime("%Y-%m-%d")
    file_path = f"{today}.csv"

    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            for row in csv.reader(f):
                if row and row[0] == name:
                    return False

    file_exists = os.path.exists(file_path)
    with open(file_path, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Name", "Time"])
        writer.writerow([name, datetime.now().strftime("%H:%M:%S")])
    return True

print("mark_attendance ready")

mark_attendance ready


In [3]:
import requests
import threading

LAPTOP_IP = "192.168.2.1"

def send_result(name, box):
    # run the network call in a background thread so recognition doesn't pause and wait
    def _send():
        try:
            requests.post(
                f"http://{LAPTOP_IP}:5000/update_result",
                json={"name": name, "box": list(box)},
                timeout=1
            )
        except Exception:
            pass
    threading.Thread(target=_send, daemon=True).start()

In [ ]:
import time
import threading

# --- background thread: continuously reads frames, always keeps only the latest ---
latest_frame = None
frame_lock = threading.Lock()
stop_reading = False

def frame_reader():
    global latest_frame
    stream = cv2.VideoCapture(f"http://{LAPTOP_IP}:5000/video")
    while not stop_reading:
        ret, frame = stream.read()
        if ret:
            with frame_lock:
                latest_frame = frame  # always overwrite, never queue up old frames
    stream.release()

reader_thread = threading.Thread(target=frame_reader, daemon=True)
reader_thread.start()
print("Frame reader started. Give it a second to connect...")
time.sleep(2)

# --- recognition loop: always works on the freshest available frame ---
candidate_name = None
candidate_since = None
VERIFY_SECONDS = 1.5

print("Running continuously. Click the \u25A0 (Interrupt) button above to stop.")

try:
    while True:
        with frame_lock:
            frame = latest_frame.copy() if latest_frame is not None else None

        if frame is None:
            continue  # no frame yet, skip this loop iteration

        # shrink a working copy for faster detection — big speedup on the board's CPU
        small = cv2.resize(frame, (640, 360))
        scale_x = frame.shape[1] / 640   # used only to scale boxes back up for the laptop display
        scale_y = frame.shape[0] / 360

        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        gray = cv2.equalizeHist(gray)  # lighting normalization, on the small frame
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(30, 30))

        # only handle the single largest face (avoids double-counting one person)
        if len(faces) > 0:
            faces = [max(faces, key=lambda f: f[2] * f[3])]
        else:
            candidate_name = None
            candidate_since = None
            send_result(None, (0, 0, 0, 0))

        for (x, y, w, h) in faces:
            # crop directly from the small grayscale image — no need to scale for recognition
            face_crop = gray[y:y+h, x:x+w]
            face_crop = cv2.resize(face_crop, (200, 200))
            label_id, confidence = recognizer.predict(face_crop)

            if confidence < CONFIDENCE_THRESHOLD:
                name = id_to_name.get(label_id, "Unknown")
            else:
                name = "Unknown"

            # scale box coordinates up to match the laptop's full-size frame for display
            box_full = (int(x*scale_x), int(y*scale_y), int(w*scale_x), int(h*scale_y))

            if name == "Unknown":
                candidate_name = None
                candidate_since = None
                send_result("Unknown", box_full)
                continue

            if name != candidate_name:
                candidate_name = name
                candidate_since = time.time()

            elapsed = time.time() - candidate_since

            if elapsed >= VERIFY_SECONDS:
                marked = mark_attendance(name)
                if marked:
                    print(f"Attendance marked: {name}")
                send_result(f"{name} - Attendance Marked!", box_full)
            else:
                send_result(None, (0, 0, 0, 0))

except KeyboardInterrupt:
    pass
finally:
    stop_reading = True
    print("Stopped.")

Frame reader started. Give it a second to connect...
Running continuously. Click the ■ (Interrupt) button above to stop.


In [ ]:
from datetime import datetime
print(datetime.now())